# 01｜只用远端现货生成最终八列表

本 Notebook 只读取一份远端原始现货文件。三状态、非零反转、零段反转、大涨和大跌均由包内冻结代码生成；不读取本地结果，不读取成交或评价记录。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

package_root = Path(os.environ.get('FINAL_UPLOAD_PACKAGE_ROOT', '/home/hzy/cta/最终冻结运行上传包_含零段反转_20260820_1350')).expanduser()
if not package_root.is_absolute():
    raise ValueError('FINAL_UPLOAD_PACKAGE_ROOT 必须是绝对路径')
package_root = package_root.resolve()
spot_raw = os.environ.get('COMPANY_SPOT_PATH', '/home/hzy/cta/IC数据更新*最终固化版/现货最终版/CSI500_SPOT_md_eod_raw*最终版.parquet')
spot = Path(spot_raw).expanduser()
if not spot.is_absolute():
    raise ValueError('COMPANY_SPOT_PATH 必须是绝对路径')
output = Path(os.environ.get('UPLOAD_OUTPUT_DIR', str(package_root / 'runtime_outputs'))).expanduser()
if not output.is_absolute():
    raise ValueError('UPLOAD_OUTPUT_DIR 必须是绝对路径')
output = output.resolve()
command = [sys.executable, str(package_root / 'src' / 'generate_compact_output.py'), '--spot', str(spot), '--output', str(output)]
subprocess.run(command, cwd=str(package_root), check=True)
print('八列表已生成：', output / '最终执行日简表.csv')

In [ ]:
import pandas as pd
import json
result_path = output / '最终执行日简表.csv'
record_path = output / '最终执行日简表_生成记录.json'
result = pd.read_csv(result_path)
record = json.loads(record_path.read_text(encoding='utf-8'))
expected_columns = ['实际执行日', '三状态', '+1反转', '-1反转', '0转-1', '0转+1', '大涨', '大跌']
assert result.columns.tolist() == expected_columns
assert record['date_mapping']['zero_transfer_formation_to_execution_exact']
display(result.tail(10))
print('执行日范围：', result['实际执行日'].min(), '->', result['实际执行日'].max())
print('最新形成日→执行日：', record['date_mapping']['latest_formation_to_execution'])